# AIDEN — Multimodal Inference on Google Colab (Free GPU)

This notebook loads **LLaVA 1.6 Mistral 7B** on a free T4 GPU and exposes a public API endpoint via **ngrok**.
Your local AIDEN backend will proxy multimodal requests here automatically.

⏱️ **Setup time:** ~5 minutes
🚀 **Once running:** Works until Colab runtime disconnects

---
## Step 1: Install Dependencies

This installs PyTorch with CUDA, HuggingFace Transformers, bitsandbytes for 4-bit quantization, and ngrok.

In [ ]:
print("Installing dependencies...")
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install -q transformers==4.47.1 accelerate bitsandbytes pillow
!pip install -q fastapi uvicorn pyngrok python-multipart
!pip install -q sentencepiece protobuf
print("✅ Dependencies installed")

---
## Step 2: Verify GPU

Colab provides a free **NVIDIA T4 GPU** (16GB VRAM). Let's confirm it's active.

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print("✅ GPU ready for LLaVA 7B")
else:
    print("❌ No GPU detected — go to Runtime > Change runtime type > T4 GPU")

---
## Step 3: Load LLaVA Model

This loads the 7B vision-language model with **4-bit quantization** (~4GB VRAM).
It takes about 30–60 seconds.

In [ ]:
import torch
from transformers import (
    LlavaNextProcessor,
    LlavaNextForConditionalGeneration,
    BitsAndBytesConfig,
)
from PIL import Image
from io import BytesIO
import base64
import time

MODEL_ID = "llava-hf/llava-v1.6-mistral-7b-hf"

print(f"Loading {MODEL_ID}...")
start = time.time()

# 4-bit quantization for memory efficiency
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

processor = LlavaNextProcessor.from_pretrained(MODEL_ID)
model = LlavaNextForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

elapsed = time.time() - start
print(f"✅ Model loaded in {elapsed:.1f}s")

# Quick sanity check
print(f"Model device: {model.device}")
print(f"Memory used: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

---
## Step 4: Test Inference (Optional)

Run a quick test to verify the model works before exposing it via API.

In [ ]:
def analyze_image(image_bytes: bytes, prompt: str = "Describe this data pipeline diagram.") -> str:
    """Run LLaVA inference on an image."""
    image = Image.open(BytesIO(image_bytes))

    conversation = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    inputs = processor.apply_chat_template(
        conversation,
        tokenize=True,
        return_tensors="pt",
        add_generation_prompt=True,
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.7,
            do_sample=True,
            pad_token_id=processor.tokenizer.eos_token_id,
        )

    response = processor.decode(outputs[0], skip_special_tokens=True)
    if "assistant" in response:
        response = response.split("assistant")[-1].strip()
    return response

# Test with a simple colored image
print("Running quick inference test...")
import io
test_img = Image.new("RGB", (512, 320), color="#1A2736")
img_bytes = io.BytesIO()
test_img.save(img_bytes, format="PNG")
img_bytes.seek(0)

result = analyze_image(img_bytes.getvalue(), "What color is this image?")
print(f"Inference result: {result[:200]}")
print("✅ Model is working!")

---
## Step 5: Expose via ngrok

ngrok creates a public HTTPS URL that tunnels to this Colab runtime.
Your local AIDEN backend will send multimodal requests here.

> **Note:** You'll need an ngrok auth token. Get one free at https://dashboard.ngrok.com/signup

In [ ]:
import os
from pyngrok import ngrok

# 👇 Paste your ngrok auth token here
# Get it from: https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN"  # ← CHANGE THIS

if NGROK_AUTH_TOKEN and NGROK_AUTH_TOKEN != "YOUR_NGROK_AUTH_TOKEN":
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("✅ ngrok auth token set")
else:
    print("⚠️  No ngrok auth token set. Run this cell again after adding your token.")

---
## Step 6: Start the Multimodal API Server

This starts a FastAPI server that accepts image uploads and runs LLaVA inference.
ngrok will tunnel it to a public URL.

In [ ]:
from fastapi import FastAPI, UploadFile, File, HTTPException
from pydantic import BaseModel
from typing import Optional
import uvicorn
import threading
import time
import requests

# ── FastAPI App ─────────────────────────────────────────────────────
app = FastAPI(title="AIDEN Multimodal Inference")


class AnalyzeRequest(BaseModel):
    image: str  # Base64 data URL
    prompt: Optional[str] = None
    temperature: Optional[float] = 0.7
    max_tokens: Optional[int] = 512


@app.get("/health")
async def health():
    return {
        "status": "healthy",
        "service": "AIDEN Multimodal",
        "model": MODEL_ID,
        "device": str(model.device),
    }


@app.post("/analyze")
async def analyze_diagram(request: AnalyzeRequest):
    """Analyze a diagram from a base64 image."""
    try:
        # Decode image
        if request.image.startswith("data:image"):
            image_data = request.image.split(",")[1]
        else:
            image_data = request.image
        image_bytes = base64.b64decode(image_data)

        prompt = request.prompt or "Describe this data pipeline diagram in detail."
        response = analyze_image(image_bytes, prompt)

        return {
            "success": True,
            "analysis": response,
            "model": MODEL_ID,
            "prompt": prompt,
            "tokens": len(response.split()),
        }
    except Exception as e:
        return {"success": False, "error": str(e)}


@app.post("/upload")
async def upload_and_analyze(file: UploadFile = File(...), prompt: Optional[str] = None):
    """Upload an image file and analyze it."""
    try:
        contents = await file.read()
        prompt_text = prompt or "Describe this data pipeline diagram in detail."
        response = analyze_image(contents, prompt_text)

        return {
            "success": True,
            "analysis": response,
            "model": MODEL_ID,
            "prompt": prompt_text,
            "tokens": len(response.split()),
        }
    except Exception as e:
        return {"success": False, "error": str(e)}


@app.get("/status")
async def status():
    return {
        "available": True,
        "model": MODEL_ID,
        "loaded": True,
    }


# ── Start server in background thread ──────────────────────────────
def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")


server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()
time.sleep(2)
print("✅ API server started on http://localhost:8000")

# ── Start ngrok tunnel ─────────────────────────────────────────────
if NGROK_AUTH_TOKEN and NGROK_AUTH_TOKEN != "YOUR_NGROK_AUTH_TOKEN":
    public_url = ngrok.connect(8000, "http")
    print(f"\n{'='*60}")
    print(f"  🔗 PUBLIC URL: {public_url}")
    print(f"{'='*60}")
    print("\nCopy this URL — you'll need it for the next step!")
else:
    print("\n⚠️  ngrok not configured. Set NGROK_AUTH_TOKEN above and re-run this cell.")
    print("   Or use the local URL: http://localhost:8000")

---
## Step 7: Verify Server Is Running

Quick test to confirm the API is responding.

In [ ]:
import requests

# Determine the public URL
if NGROK_AUTH_TOKEN and NGROK_AUTH_TOKEN != "YOUR_NGROK_AUTH_TOKEN":
    tunnels = ngrok.get_tunnels()
    if tunnels:
        BASE_URL = tunnels[0].public_url
    else:
        BASE_URL = "http://localhost:8000"
else:
    BASE_URL = "http://localhost:8000"

print(f"Testing endpoint: {BASE_URL}")

# Test health
r = requests.get(f"{BASE_URL}/health")
print(f"Health: {r.json()}")

# Test status
r = requests.get(f"{BASE_URL}/status")
print(f"Status: {r.json()}")

print(f"\n{'='*60}")
print(f"  ✅ Multimodal API is live at:")
print(f"  🔗 {BASE_URL}")
print(f"{'='*60}")
print(f"\n📋 Next: Configure your local AIDEN backend to use this URL.")

---
## Step 8: Configure Local Backend

On your local machine (where AIDEN backend runs), set this environment variable:

```bash
# Replace with your actual ngrok URL from above
export MULTIMODAL_REMOTE_URL=https://abc123.ngrok-free.app

# Also enable multimodal
export MULTIMODAL_ENABLED=True

# Restart the backend
cd backend
uvicorn app.main:app --reload --port 8000
```

Your local backend will now proxy all multimodal requests to this Colab notebook.
Keep this notebook running while you test!

---
## Step 9: Test from Local Machine

Once configured, run this from your **local terminal**:

```bash
# 1. Get a JWT token
TOKEN=$(curl -s -X POST http://localhost:8000/api/v1/auth/login \
  -H "Content-Type: application/x-www-form-urlencoded" \
  -d "username=demo@example.com&password=demo1234" \
  | python -c "import sys,json; print(json.load(sys.stdin)['access_token'])")

# 2. Check multimodal status
curl http://localhost:8000/api/v1/multimodal/status \
  -H "Authorization: Bearer $TOKEN"

# 3. Upload a diagram
curl -X POST http://localhost:8000/api/v1/multimodal/upload \
  -H "Authorization: Bearer $TOKEN" \
  -F "file=@backend/data/training/diagram_0000.png" \
  -F "prompt=Describe this data pipeline architecture"
```

Or use the frontend at **http://localhost:5174/multimodal**.

---
## ⏱️ Keep Colab Alive

Colab disconnects after ~90 minutes of inactivity. To keep it running:

1. **Keep this tab open** in your browser
2. **Run this cell** to auto-click every 60 seconds:

```javascript
function ClickConnect(){
  console.log("Keeping Colab alive...");
  document.querySelector("colab-connect-button")?.click();
}
setInterval(ClickConnect, 60000);
```

3. Or use a service like **Colab-Alive** browser extension

If disconnected, just re-run the notebook — it takes ~2 minutes to reload.

---
## 🔧 Troubleshooting

| Problem | Fix |
|---------|-----|
| **Out of memory** | Restart runtime: Runtime → Factory reset runtime. Then skip the test inference cell. |
| **ngrok auth failed** | Make sure you copied the full token from https://dashboard.ngrok.com/get-started/your-authtoken |
| **No GPU** | Go to Runtime → Change runtime type → T4 GPU |
| **Model download slow** | It's ~15GB. Colab has fast internet (~50 MB/s) — should take ~5 minutes. |
| **Local backend can't connect** | Make sure `MULTIMODAL_REMOTE_URL` is set correctly (no trailing slash) |
| **CORS errors** | The Colab server has CORS wide open — if needed, add your local URL to the allow list. |
| **408 Request Timeout** | LLaVA inference takes 5-15s on T4. Make sure your local timeout is ≥30s. |

---
*Notebook generated for AIDEN — AI Data Engineering*